In [ ]:
# --- repo bootstrap -------------------------------------------------------
# Resolves the repository root and chdirs to it, so every path below is
# repo-relative and this notebook runs from any checkout location.
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)


In [1]:
from ultralytics import YOLO
import os

In [2]:
person_detection_model = YOLO(model="models/person/yolo11l_ep200/best.pt")

In [3]:
os.makedirs('data/interim/resized_quality_images', exist_ok=True)

In [4]:
#resize image to 640x640
from PIL import Image

os.makedirs('data/interim/resized_quality_images', exist_ok=True)

def resize_image(input_path):
    # Open the source image
    img = Image.open(input_path)

    # Resize to an exact 640x640 pixel square
    resized_img = img.resize((640, 640))

    # Save the newly sized file
    output_path = os.path.join('data/interim/resized_quality_images', f"resized_{os.path.basename(input_path)}")
    resized_img.save(output_path)

In [5]:
for image in os.listdir("data/external/images/set1"):
    image_path = os.path.join("data/external/images/set1", image)
    resize_image(image_path)

In [9]:
#Run batched inference on a list of images
results = person_detection_model.predict(source="data/interim/resized_quality_images", imgsz=640, classes=[0,1], conf=0.5, save=True,
                                          project=str(ROOT / "outputs/predictions"), name="person_detect")



image 1/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_1.jpg: 640x640 2 half_persons, 4.4ms
image 2/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_10.jpg: 640x640 2 persons, 4.0ms
image 3/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_11.jpg: 640x640 2 half_persons, 3.9ms
image 4/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_12.jpg: 640x640 2 persons, 3.8ms
image 5/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_13.jpg: 640x640 3 half_persons, 1 person, 3.8ms
image 6/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_14.jpg: 640x640 1 half_person, 1 person, 3.8ms
image 7/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_15.jpg: 640x640 1 half_person, 3.9ms
image 8/16 /home/ai_vison/Desktop/PPE/model_pipeline/resized_quality_images/resized_16.jpg: 640x640 1 half_person, 4 persons, 3.8ms
image 

In [10]:
len(results)

16

In [11]:
import torchvision

def calculate_iou(box1, box2):
    # box_iou compares all boxes in box1 to all boxes in box2
    iou = torchvision.ops.box_iou(box1, box2)
    return iou

In [12]:
def is_overlap(box1, box2):
    iou = calculate_iou(box1, box2)
    # print(iou)
    return iou > 0.7

def non_max_suppression(boxes):
    # Sort boxes by confidence in descending order
    try: 
        boxes = sorted(boxes, key=lambda x: x.conf, reverse=True)
        keep = []

        while boxes:
            # Take the box with highest confidence
            current = boxes.pop(0)
            keep.append(current)

            # Remove boxes that overlap too much with current box
            boxes = [box for box in boxes if
                        not is_overlap(current.xyxy, box.xyxy)]

        return keep
    except Exception as e:
        print(f"An error occurred during non-max suppression: {e}")
        return []

In [14]:
import os
import cv2

# Create a directory to save the extracted images
save_dir = "data/interim/extracted_humans"
os.makedirs(save_dir, exist_ok=True)

classes = ['half_person', 'person']

def save_extracted_objects(keep, inf_img, img_index):
    for i, k in enumerate(keep):
        # Extract coordinates and convert them to integers
        x1, y1, x2, y2 = map(int, k.xyxy[0].tolist())

        cls = list(k.cls)[0].item()

        # Crop from the original BGR image
        cropped_bgr = inf_img[0].orig_img[y1:y2, x1:x2]
        
        # Create a filename with the index and confidence score
        filename = os.path.join(save_dir, f"human_{classes[int(cls)]}_{img_index}_{i}.jpg")
        
        # Save the image to disk
        cv2.imwrite(filename, cropped_bgr)
        print(f"Saved: {filename}")

In [16]:
for j, inf_img in enumerate(results):
    keep = non_max_suppression(inf_img.boxes)
    save_extracted_objects(keep, inf_img, j)

Saved: extracted_humans/human_half_person_0_0.jpg
Saved: extracted_humans/human_half_person_0_1.jpg
Saved: extracted_humans/human_person_1_0.jpg
Saved: extracted_humans/human_person_1_1.jpg
Saved: extracted_humans/human_half_person_2_0.jpg
Saved: extracted_humans/human_half_person_2_1.jpg
Saved: extracted_humans/human_person_3_0.jpg
Saved: extracted_humans/human_person_3_1.jpg
Saved: extracted_humans/human_half_person_4_0.jpg
Saved: extracted_humans/human_half_person_4_1.jpg
Saved: extracted_humans/human_half_person_4_2.jpg
Saved: extracted_humans/human_person_5_0.jpg
Saved: extracted_humans/human_half_person_6_0.jpg
Saved: extracted_humans/human_person_7_0.jpg
Saved: extracted_humans/human_person_7_1.jpg
Saved: extracted_humans/human_person_7_2.jpg
Saved: extracted_humans/human_person_7_3.jpg
Saved: extracted_humans/human_half_person_8_0.jpg
Saved: extracted_humans/human_half_person_9_0.jpg
Saved: extracted_humans/human_half_person_9_1.jpg
Saved: extracted_humans/human_half_person_10_

In [17]:
ppe_detection_model = YOLO(model="models/ppe/yolo11l_ep200/best.pt")

In [18]:
#resize image to 640x640
from PIL import Image

os.makedirs('data/interim/resized_human_images', exist_ok=True)

def resize_image_2(input_path):
    # Open the source image
    img = Image.open(input_path)

    # width, height
    resized_img = img.resize((171, 455))

    # Save the newly sized file
    output_path = os.path.join('data/interim/resized_human_images', f"resized_{os.path.basename(input_path)}")
    resized_img.save(output_path)

In [19]:
for image in os.listdir("data/interim/extracted_humans"):
    image_path = os.path.join("data/interim/extracted_humans", image)
    resize_image_2(image_path)

In [20]:
# (height, width)
results_ppe = ppe_detection_model.predict(source="data/interim/resized_human_images", 
                                          imgsz=[455, 171], 
                                          classes=[0,1,2], 
                                          conf=0.5, save=True, 
                                          project=str(ROOT / "outputs/predictions"),
                                          name="ppe_results"
                                          )

# return a list of Results objects


WARNING ⚠️ imgsz=[455, 171] must be multiple of max stride 32, updating to [480, 192]
image 1/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_0_0.jpg: 480x192 1 helmet, 1 jacket, 4.4ms
image 2/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_0_1.jpg: 480x192 1 helmet, 1 jacket, 4.1ms
image 3/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_10_0.jpg: 480x192 1 helmet, 3.9ms
image 4/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_10_1.jpg: 480x192 1 jacket, 4.2ms
image 5/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_12_0.jpg: 480x192 1 helmet, 1 jacket, 4.0ms
image 6/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_images/resized_human_half_person_12_1.jpg: 480x192 1 helmet, 1 jacket, 4.0ms
image 7/30 /home/ai_vison/Desktop/PPE/model_pipeline/resized_human_imag

In [16]:
# ['boot', 'helmet', 'jacket']